In [2]:
%env FS_LICENSE=/meeg-pipelines/license.txt

env: FS_LICENSE=/meeg-pipelines/license.txt


In [14]:
import mne

subject_id = "10"
subject = f"sub-{subject_id}"

subjects_dir = r"D:\VM\derivatives\test\sourcedata\freesurfer"


forward_path = fr"D:\VM\derivatives\test\sub-{subject_id}\ses-01\meg\forward.fif"

forward = mne.read_forward_solution(forward_path)

Reading forward solution from D:\VM\derivatives\test\sub-10\ses-01\meg\forward.fif...
    Reading a source space...


C:\Users\ferat\AppData\Local\Temp\ipykernel_14076\1547628357.py:11: RuntimeWarning: This filename (D:\VM\derivatives\test\sub-10\ses-01\meg\forward.fif) does not conform to MNE naming conventions. All forward files should end with -fwd.fif, -fwd.fif.gz, _fwd.fif, _fwd.fif.gz, -fwd.h5 or _fwd.h5
  forward = mne.read_forward_solution(forward_path)


    [done]
    1 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read MEG forward solution (3727 sources, 306 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame


In [15]:
src = forward['src']
trans = forward['mri_head_t']

fig = mne.viz.plot_alignment(src=src, subject=subject, trans=trans, subjects_dir=subjects_dir)


For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
Using outer_skin.surf for head surface.


In [2]:
import struct
import numpy as np
import os

class DFSReader:
    """Reader for BrainSuite DFS (surface) files"""
    
    def __init__(self, filepath):
        self.filepath = filepath
        self.vertices = None
        self.faces = None
        self.normals = None
        self.colors = None
        self.uv = None
        self.labels = None
        self.attributes = None
        self._read_dfs()
    
    def _read_dfs(self):
        """Read DFS file structure"""
        with open(self.filepath, 'rb') as f:
            # Read header signature
            sig = f.read(12).decode('ascii').strip('\x00')
            if not sig.startswith('DFS'):
                raise ValueError(f"Invalid DFS file signature: {sig}")
            
            # Read header information
            hdrsize = struct.unpack('<i', f.read(4))[0]
            mdoffset = struct.unpack('<i', f.read(4))[0]
            pdoffset = struct.unpack('<i', f.read(4))[0]
            nTriangles = struct.unpack('<i', f.read(4))[0]
            nVertices = struct.unpack('<i', f.read(4))[0]
            nStrips = struct.unpack('<i', f.read(4))[0]
            stripSize = struct.unpack('<i', f.read(4))[0]
            normals = struct.unpack('<i', f.read(4))[0]
            uvStart = struct.unpack('<i', f.read(4))[0]
            vcoffset = struct.unpack('<i', f.read(4))[0]
            labelOffset = struct.unpack('<i', f.read(4))[0]
            vertexAttributes = struct.unpack('<i', f.read(4))[0]
            
            print(f"Debug - Header size: {hdrsize}")
            print(f"Debug - nTriangles: {nTriangles}, nVertices: {nVertices}")
            print(f"Debug - Current position after header: {f.tell()}")
            
            # The actual data starts right after the header at position 184
            # Header is 172 bytes (12 byte signature + 160 bytes of header data)
            # Position should be at 184 after reading header
            data_start = 184
            
            # Seek to triangle data start
            f.seek(data_start)
            
            # Read triangle data
            face_data = np.frombuffer(f.read(nTriangles * 3 * 4), dtype=np.int32)
            self.faces = face_data.reshape((nTriangles, 3))
            
            print(f"Debug - Position after reading faces: {f.tell()}")
            
            # Read vertex data
            vertex_data = np.frombuffer(f.read(nVertices * 3 * 4), dtype=np.float32)
            self.vertices = vertex_data.reshape((nVertices, 3))
            
            print(f"Debug - Position after reading vertices: {f.tell()}")
            
            # Read optional data fields
            if normals > 0:
                normal_data = np.frombuffer(f.read(nVertices * 3 * 4), dtype=np.float32)
                self.normals = normal_data.reshape((nVertices, 3))
            
            if vcoffset > 0:
                color_data = np.frombuffer(f.read(nVertices * 3 * 4), dtype=np.float32)
                self.colors = color_data.reshape((nVertices, 3))
            
            if uvStart > 0:
                uv_data = np.frombuffer(f.read(nVertices * 2 * 4), dtype=np.float32)
                self.uv = uv_data.reshape((nVertices, 2))
            
            if labelOffset > 0:
                self.labels = np.frombuffer(f.read(nVertices * 2), dtype=np.int16)
            
            if vertexAttributes > 0:
                self.attributes = np.frombuffer(f.read(nVertices * 4), dtype=np.float32)
    
    def to_freesurfer_surface(self, output_path):
        """Export to FreeSurfer surface format (e.g., lh.pial, rh.white)"""
        TRIANGLE_FILE_MAGIC_NUMBER = 16777214
        
        with open(output_path, 'wb') as f:
            # Write magic number
            f.write(struct.pack('>3B', 255, 255, 254))
            
            # Write comment (created on line, blank line)
            comment = f"Created from {os.path.basename(self.filepath)}\n\n"
            f.write(comment.encode('utf-8'))
            
            # Write number of vertices and faces
            f.write(struct.pack('>I', len(self.vertices)))
            f.write(struct.pack('>I', len(self.faces)))
            
            # Write vertices
            for vertex in self.vertices:
                f.write(struct.pack('>fff', vertex[0], vertex[1], vertex[2]))
            
            # Write faces
            for face in self.faces:
                f.write(struct.pack('>iii', face[0], face[1], face[2]))
        
        print(f"Exported FreeSurfer surface to: {output_path}")
    
    def to_freesurfer_curv(self, output_path, data=None):
        """Export to FreeSurfer curvature format (.curv)"""
        if data is None:
            data = self.attributes
        
        if data is None:
            raise ValueError("No attribute data available to export as curvature")
        
        NEW_VERSION_MAGIC_NUMBER = 16777215
        
        with open(output_path, 'wb') as f:
            # Write magic number
            f.write(struct.pack('>3B', 255, 255, 255))
            
            # Write number of vertices and faces
            f.write(struct.pack('>I', len(self.vertices)))
            f.write(struct.pack('>I', len(self.faces)))
            
            # Write number of values per vertex (always 1 for curvature)
            f.write(struct.pack('>I', 1))
            
            # Write data
            for val in data:
                f.write(struct.pack('>f', val))
        
        print(f"Exported FreeSurfer curvature to: {output_path}")
    
    def to_freesurfer_annot(self, output_path):
        """Export to FreeSurfer annotation format (.annot)"""
        if self.labels is None:
            raise ValueError("No label data available to export as annotation")
        
        # Get unique labels
        unique_labels = np.unique(self.labels)
        
        # Create color table (random colors for each label)
        np.random.seed(42)
        color_table = {}
        for label in unique_labels:
            r, g, b = np.random.randint(0, 256, 3)
            color_table[int(label)] = (r, g, b, 255, int(r + g*256 + b*65536))
        
        with open(output_path, 'wb') as f:
            # Write number of vertices
            f.write(struct.pack('>I', len(self.labels)))
            
            # Write vertex indices and labels
            for i, label in enumerate(self.labels):
                r, g, b, a, packed = color_table[int(label)]
                f.write(struct.pack('>I', i))
                f.write(struct.pack('>i', packed))
            
            # Write color table
            f.write(struct.pack('>I', 1))  # Has color table
            f.write(struct.pack('>I', len(unique_labels)))  # Number of entries
            
            # Original filename length
            orig_name = "converted_from_dfs"
            f.write(struct.pack('>I', len(orig_name)))
            f.write(orig_name.encode('utf-8'))
            
            # Number of entries again
            f.write(struct.pack('>I', len(unique_labels)))
            
            # Write each entry
            for label in unique_labels:
                r, g, b, a, packed = color_table[int(label)]
                # Structure index
                f.write(struct.pack('>I', int(label)))
                # Name
                name = f"label_{label}"
                f.write(struct.pack('>I', len(name)))
                f.write(name.encode('utf-8'))
                # RGBA
                f.write(struct.pack('>4i', r, g, b, a))
        
        print(f"Exported FreeSurfer annotation to: {output_path}")
    
    def export_all(self, output_prefix):
        """Export all available data to FreeSurfer formats"""
        # Always export surface
        self.to_freesurfer_surface(f"{output_prefix}.surf")
        
        # Export curvature if available
        if self.attributes is not None:
            self.to_freesurfer_curv(f"{output_prefix}.curv")
        
        # Export annotation if available
        if self.labels is not None:
            self.to_freesurfer_annot(f"{output_prefix}.annot")
        
        print(f"\nExport complete! Files created with prefix: {output_prefix}")


# Example usage
if __name__ == "__main__":
    import sys
    
    if len(sys.argv) < 2:
        print("Usage: python dfs_to_freesurfer.py <input.dfs> [output_prefix]")
        print("\nExample:")
        print("  python dfs_to_freesurfer.py brain.dfs lh.pial")
        print("\nThis will create:")
        print("  - lh.pial.surf (surface geometry)")
        print("  - lh.pial.curv (curvature, if available)")
        print("  - lh.pial.annot (labels, if available)")
        sys.exit(1)
    
    input_file = sys.argv[1]
    output_prefix = sys.argv[2] if len(sys.argv) > 2 else "output"
    
    # Read DFS file
    print(f"Reading DFS file: {input_file}")
    dfs = DFSReader(input_file)
    
    print(f"\nDFS file info:")
    print(f"  Vertices: {len(dfs.vertices)}")
    print(f"  Faces: {len(dfs.faces)}")
    print(f"  Has normals: {dfs.normals is not None}")
    print(f"  Has colors: {dfs.colors is not None}")
    print(f"  Has UV: {dfs.uv is not None}")
    print(f"  Has labels: {dfs.labels is not None}")
    print(f"  Has attributes: {dfs.attributes is not None}")
    
    # Export all available data
    print(f"\nExporting to FreeSurfer format...")
    dfs.export_all(output_prefix)
    

Reading DFS file: -f


FileNotFoundError: [Errno 2] No such file or directory: '-f'

In [27]:
input_file = r"D:\brainsuite.brain.dfs"
dfs = DFSReader(input_file)
dfs.to_freesurfer_surface(r"D:\brain_surf.surf")

Debug - Header size: 184
Debug - nTriangles: 215704, nVertices: 107854
Debug - Current position after header: 60
Debug - Position after reading faces: 2588632
Debug - Position after reading vertices: 3882880
Exported FreeSurfer surface to: D:\brain_surf.surf


In [28]:
input_file = r"D:\brainsuite.inner_skull.dfs"
dfs = DFSReader(input_file)
dfs.to_freesurfer_surface(r"D:\\brainsuite.inner_skull.surf")

Debug - Header size: 184
Debug - nTriangles: 211052, nVertices: 105528
Debug - Current position after header: 60
Debug - Position after reading faces: 2532808
Debug - Position after reading vertices: 3799144
Exported FreeSurfer surface to: D:\\brainsuite.inner_skull.surf


In [30]:
input_file = r"D:\brainsuite.scalp.dfs"
dfs = DFSReader(input_file)
dfs.to_freesurfer_surface(r"D:\\brainsuite.scalp.surf")

Debug - Header size: 184
Debug - nTriangles: 538964, nVertices: 269262
Debug - Current position after header: 60
Debug - Position after reading faces: 6467752
Debug - Position after reading vertices: 9698896
Exported FreeSurfer surface to: D:\\brainsuite.scalp.surf


In [ ]:
input_file = r"D:\brainsuite.outer_skull.dfs"
dfs = DFSReader(input_file)
dfs.to_freesurfer_surface(r"D:\\bouter_skull.surf")

Debug - Header size: 184
Debug - nTriangles: 231144, nVertices: 115574
Debug - Current position after header: 60
Debug - Position after reading faces: 2773912
Debug - Position after reading vertices: 4160800
Exported FreeSurfer surface to: D:\\brainsuite.outer_skull.surf


In [21]:
src.plot(trans=trans, brain='white', skull=True, subjects_dir=subjects_dir)

For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


In [6]:
import mne


subjects_dir = r"/data/BrainSuite"
subject = "sub-test"

bem_folder = os.path.join(subjects_dir, subject, 'bem')

brainsuite_brain_path = os.path.join(bem_folder, 'brainsuite.brain.dfs')
freesurfer_brain_path = os.path.join(bem_folder, 'brain.surf')
dfs = DFSReader(brainsuite_brain_path)
dfs.to_freesurfer_surface(freesurfer_brain_path)

brainsuite_outer_skull_path = os.path.join(bem_folder, 'brainsuite.outer_skull.dfs')
freesurfer_outer_skull_path = os.path.join(bem_folder, 'outer_skull.surf')
dfs = DFSReader(brainsuite_outer_skull_path)
dfs.to_freesurfer_surface(freesurfer_outer_skull_path)

brainsuite_inner_skull_path = os.path.join(bem_folder, 'brainsuite.inner_skull.dfs')
freesurfer_inner_skull_path = os.path.join(bem_folder, 'inner_skull.surf')
dfs = DFSReader(brainsuite_inner_skull_path)
dfs.to_freesurfer_surface(freesurfer_inner_skull_path)

brainsuite_scalp_path = os.path.join(bem_folder, 'brainsuite.scalp.dfs')
freesurfer_scalp_path = os.path.join(bem_folder, 'outer_skin.surf')
dfs = DFSReader(brainsuite_scalp_path)
dfs.to_freesurfer_surface(freesurfer_scalp_path)

"""
n_triangles = 20480
method = 'sphere'

for surface in [freesurfer_brain_path, freesurfer_outer_skull_path, freesurfer_inner_skull_path, freesurfer_scalp_path]:
        mesh = mne.read_surface(surface, read_metadata=True)
        decimated_mesh = mne.decimate_surface(
            mesh[0], mesh[1], n_triangles=n_triangles, method=method
        )
        mne.write_surface(
            surface,
            decimated_mesh[0],
            decimated_mesh[1],
            volume_info=mesh[2],
            overwrite=True,
        )
"""
        
mne.bem.make_bem_model(subject=subject, conductivity=[0.3, 0.006, 0.3], subjects_dir=subjects_dir)

Debug - Header size: 184
Debug - nTriangles: 202912, nVertices: 101458
Debug - Current position after header: 60
Debug - Position after reading faces: 2435128
Debug - Position after reading vertices: 3652624
Exported FreeSurfer surface to: /data/BrainSuite/sub-test/bem/brain.surf
Debug - Header size: 184
Debug - nTriangles: 225040, nVertices: 112522
Debug - Current position after header: 60
Debug - Position after reading faces: 2700664
Debug - Position after reading vertices: 4050928
Exported FreeSurfer surface to: /data/BrainSuite/sub-test/bem/outer_skull.surf
Debug - Header size: 184
Debug - nTriangles: 197388, nVertices: 98696
Debug - Current position after header: 60
Debug - Position after reading faces: 2368840
Debug - Position after reading vertices: 3553192
Exported FreeSurfer surface to: /data/BrainSuite/sub-test/bem/inner_skull.surf
Debug - Header size: 184
Debug - nTriangles: 482520, nVertices: 241062
Debug - Current position after header: 60
Debug - Position after reading fa

RuntimeError: Cannot decimate to requested ico grade 4. The provided BEM surface has 197388 triangles, which cannot be isomorphic with a subdivided icosahedron. Consider manually decimating the surface to a suitable density and then use ico=None in make_bem_model.

In [1]:
from meegpype.interfaces.mne.report import MakeReport

In [3]:
subjects_dir = r"/data/VM/derivatives/test/sourcedata/freesurfer"
subject_id = "sub-07"
forward = r"/data/VM/derivatives/test/sub-07/ses-01/meg/forward.fif"
info = r"/data/VM/derivatives/test/sub-07/ses-01/meg/sub-07_ses-01_task-hippo_run-01_proc-tsss_meg.fif"
bem = r"/data/VM/derivatives/test"


make_report = MakeReport()
make_report.inputs.subjects_dir = subjects_dir
make_report.inputs.subject = subject_id
make_report.inputs.forward = forward
make_report.inputs.fname = '/data/test_report'
make_report.inputs.info = info
make_report.inputs.overwrite = True

make_report.run()

/opt/conda/envs/smriprep/lib/python3.11/site-packages/pyvista/plotting/utilities/xvfb.py:48: PyVistaDeprecationWarning: This function is deprecated and will be removed in future version of PyVista. Use vtk-osmesa instead.
  warnings.warn(


Embedding : jquery-3.6.0.min.js
Embedding : bootstrap.bundle.min.js
Embedding : bootstrap.min.css
Embedding : bootstrap-table/bootstrap-table.min.js
Embedding : bootstrap-table/bootstrap-table.min.css
Embedding : bootstrap-table/bootstrap-table-copy-rows.min.js
Embedding : bootstrap-table/bootstrap-table-export.min.js
Embedding : bootstrap-table/tableExport.min.js
Embedding : bootstrap-icons/bootstrap-icons.mne.min.css
Embedding : highlightjs/highlight.min.js
Embedding : highlightjs/atom-one-dark-reasonable.min.css
Reading forward solution from /data/VM/derivatives/test/sub-07/ses-01/meg/forward.fif...
    Reading a source space...


/meeg-pipelines/meegpype/interfaces/mne/report.py:119: RuntimeWarning: This filename (/data/VM/derivatives/test/sub-07/ses-01/meg/forward.fif) does not conform to MNE naming conventions. All forward files should end with -fwd.fif, -fwd.fif.gz, _fwd.fif, _fwd.fif.gz, -fwd.h5 or _fwd.h5
  report.add_forward(


    [done]
    1 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read MEG forward solution (3516 sources, 306 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
Using outer_skin.surf for head surface.
Overwriting existing file.
Saving report to : /data/test_report.html
Overwriting existing file.
Saving report to : /data/test_report.hdf5


In [ ]:
mne.viz.plot_bem(src=src, subject=subject, subjects_dir=subjects_dir, show=True)

Using surface: D:\VM\derivatives\test\sourcedata\freesurfer\sub-10\bem\inner_skull.surf
Using surface: D:\VM\derivatives\test\sourcedata\freesurfer\sub-10\bem\outer_skull.surf
Using surface: D:\VM\derivatives\test\sourcedata\freesurfer\sub-10\bem\outer_skin.surf


ValueError: Source space must be in MRI coordinates, got head